In [13]:
# !pip install arxiv
# !pip install google-genai

In [2]:
import arxiv
import json 
import os
from typing import List
from dotenv import load_dotenv

In [3]:
PAPER_DIR = "papers"

In [4]:
gemini_api_key = os.getenv('GEMINI_API_KEY')

In [5]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    client = arxiv.Client()
    search = arxiv.Search(
        query=topic,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    papers = client.results(search)
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    file_path = os.path.join(path, "papers_info.json")
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info

    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    print(f"Results are saved in {file_path}")
    return paper_ids

In [6]:
search_papers("physics")

Results are saved in papers/physics/papers_info.json


['1910.11775v2',
 'hep-ex/9605011v1',
 '2107.11742v1',
 '0710.4947v3',
 '2312.14190v1']

In [7]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
 
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."


In [8]:
print(extract_info('1910.11775v2'))

{
  "title": "Physics Briefing Book",
  "authors": [
    "European Strategy for Particle Physics Preparatory Group"
  ],
  "summary": "The European Particle Physics Strategy Update (EPPSU) process takes a bottom-up approach, whereby the community is first invited to submit proposals (also called inputs) for projects that it would like to see realised in the near-term, mid-term and longer-term future. National inputs as well as inputs from National Laboratories are also an important element of the process. All these inputs are then reviewed by the Physics Preparatory Group (PPG), whose role is to organize a Symposium around the submitted ideas and to prepare a community discussion on the importance and merits of the various proposals. The results of these discussions are then concisely summarised in this Briefing Book, prepared by the Conveners, assisted by Scientific Secretaries, and with further contributions provided by the Contributors listed on the title page. This constitutes the 

In [9]:
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

In [10]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
        
    elif isinstance(result, list):
        result = ', '.join(result)
        
    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)
    
    else:
        # For any other type, convert using str()
        result = str(result)
    return result_tool_function[tool_name](**tool_args)

    if result is None:
        result = ""

In [48]:
from google import genai
from google.genai import types

from google.genai.types import Tool
load_dotenv()
client = genai.Client()


messages = [
    {
        "role": "user",
        "parts": [
            {"text": "How are you?"}
        ]
    }
]
response = client.models.generate_content(
        model='gemini-2.5-flash', # Choose a suitable Gemini model, e.g., 'gemini-2.5-flash' or 'gemini-2.5-pro'
        contents=messages,
        config=types.GenerateContentConfig(
            max_output_tokens=2024, # Renamed from max_tokens
            # tools=tools,            # Takes a list of Tool objects
        )
    )
print(response.text)

I'm doing well, thank you for asking! As an AI, I don't experience feelings or have a physical state, but I am fully operational and ready to assist you.

How can I help you today?


In [49]:
def openai_to_gemini_messages(messages):
    """
    Convert OpenAI chat messages into Gemini-compatible
    'contents' for generate_content().
    """

    gemini_messages = []

    for m in messages:
        role = m.get("role", "user")
        content = m.get("content")
        tool_calls = m.get("tool_calls")
        tool_call_id = m.get("tool_call_id")

        # --- Role Mapping ---
        if role == "assistant":
            role = "model"
        elif role == "tool":
            role = "model"
        elif role == "system":
            # Gemini has no system role → treat as high-priority user instruction
            role = "user"

        parts = []

        # ---- PROCESS CONTENT ----

        # Case 1: Simple text string
        if isinstance(content, str) and content.strip():
            parts.append({"text": content})

        # Case 2: Array of mixed content (OpenAI v2 format)
        elif isinstance(content, list):
            for item in content:
                if isinstance(item, str):
                    parts.append({"text": item})
                    continue

                if not isinstance(item, dict):
                    parts.append({"text": str(item)})
                    continue

                t = item.get("type")

                # text part
                if t == "text":
                    parts.append({"text": item.get("text", "")})

                # image part
                elif t == "image_url":
                    url = item["image_url"]["url"]

                    # base64 inline
                    if url.startswith("data:"):
                        header, b64data = url.split(",", 1)
                        mime_type = header.split(";")[0].split(":")[1]
                        parts.append({
                            "inline_data": {
                                "mime_type": mime_type,
                                "data": b64data
                            }
                        })
                    else:
                        # non-base64 image URLs → not directly supported
                        parts.append({"text": f"[Image URL: {url}]"})
                else:
                    parts.append({"text": str(item)})

        # ---- Process Tool Calls (assistant -> tool) ----
        if tool_calls:
            for call in tool_calls:
                parts.append({
                    "tool_call": {
                        "id": call.get("id"),
                        "name": call.get("name"),
                        "arguments": call.get("arguments", "{}")
                    }
                })

        # ---- Process Tool Response Messages ----
        if role == "model" and tool_call_id and content:
            parts.append({
                "tool_response": {
                    "tool_call_id": tool_call_id,
                    "response": content
                }
            })

        # Fallback if nothing added
        if not parts:
            parts.append({"text": ""})

        gemini_messages.append({
            "role": role,
            "parts": parts
        })

    return gemini_messages


In [50]:
def process_query(query):
    # keep messages in OPENAI STYLE only
    messages = [
        {"role": "user", "content": query}
    ]

    # initial Gemini request
    gemini_messages = openai_to_gemini_messages(messages)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=gemini_messages,
        config=types.GenerateContentConfig(
            max_output_tokens=2024,
            tools=[
                Tool(function=search_papers),
                Tool(function=extract_info)
            ],
        )
    )

    # continue while tool calls occur
    process_loop = True

    while process_loop:

        # this will store the assistant text for OpenAI-style message
        openai_assistant_content = []

        for part in response.candidates[0].content.parts:

            # ---- CASE 1: MODEL RETURNS TEXT ----
            if part.type == "text":
                print(part.text)
                openai_assistant_content.append(part.text)

            # ---- CASE 2: MODEL RETURNS TOOL USAGE ----
            elif part.type == "tool_use":

                tool_call = part.tool_use

                tool_id   = tool_call.id
                tool_name = tool_call.name
                tool_args = tool_call.input

                print(f"Calling tool {tool_name} with args {tool_args}")

                # Append assistant tool_call to OpenAI-style messages
                messages.append({
                    "role": "assistant",
                    "tool_calls": [
                        {
                            "id": tool_id,
                            "name": tool_name,
                            "arguments": tool_args
                        }
                    ]
                })

                # Execute your local Python tool
                result = execute_tool(tool_name, tool_args)

                # Append user "tool result" message in OpenAI format
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_id,
                    "content": result
                })

                # Re-run Gemini
                gemini_messages = openai_to_gemini_messages(messages)

                response = client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=gemini_messages,
                    config=types.GenerateContentConfig(
                        max_output_tokens=2024,
                        tools=[
                            Tool(function=search_papers),
                            Tool(function=extract_info)
                        ],
                    )
                )

                # break out and continue main loop
                break

        else:
            # no tool use → final assistant response
            messages.append({
                "role": "assistant",
                "content": "".join(openai_assistant_content)
            })
            process_loop = False

    return messages


In [51]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

In [52]:
chat_loop()

Type your queries or 'quit' to exit.

Error: 1 validation error for Tool
function
  Extra inputs are not permitted [type=extra_forbidden, input_value=<function search_papers at 0x10a830fe0>, input_type=function]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
